[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/31_transformers/03_mini_gpt.ipynb)

# Bài 03 — Mini GPT: Transformer decoder-only từ đầu

**🎯 Mục tiêu**
- Lắp một **GPT** nhỏ: token + position embedding → các khối Transformer với **causal self-attention** → LayerNorm → đầu ra (weight tying).
- Kiểm tra **tính nhân quả**; huấn luyện mô hình ngôn ngữ ký tự trên review tiếng Việt, so với LSTM của chương 30 một cách công bằng.
- **Lấy mẫu**: nhiệt độ, top-$k$, top-$p$ (nucleus).
- Thấy Transformer mạnh ở đâu: bài toán **sao chép trong ngữ cảnh** mà LSTM không giải được.
- **KV cache** khi sinh; vài con số về quy mô (tham số, FLOPs, dữ liệu).

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/31_transformers

In [ ]:
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ml_course.visualization import set_style

try:
    import torch
    from torch import nn
except ImportError as e:
    raise ImportError("Chương này cần PyTorch: pip install -e \".[dl]\"") from e

set_style()
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.deterministic = True
print("thiết bị:", device)

## 1. Kiến trúc

GPT (Radford et al., 2018, 2019) là **chồng khối Transformer chỉ có decoder**, huấn luyện dự đoán token tiếp theo:

```
token ids ─► tok_emb + pos_emb ─► [Block: x + CausalMHA(LN x); x + FFN(LN x)] × L ─► LN ─► Linear (= tok_emb^T) ─► logits
```

- **Causal self-attention**: token $i$ chỉ nhìn các vị trí $\le i$ (mask tam giác dưới, bài 01).
- **Weight tying**: ma trận đầu ra dùng chung với embedding token (Press & Wolf, 2017) — ít tham số hơn, thường tốt hơn.
- Khởi tạo $\mathcal N(0, 0.02)$ như GPT-2. Lớp attention dưới đây có thêm **KV cache** (mục 6).

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, context):
        super().__init__()
        self.h, self.d_k = n_heads, d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.register_buffer("mask", torch.tril(torch.ones(context, context, dtype=torch.bool)))

    def forward(self, x, cache=None):
        B, T, D = x.shape
        q, k, v = self.qkv(x).split(D, dim=-1)
        q, k, v = (t.view(B, T, self.h, self.d_k).transpose(1, 2) for t in (q, k, v))
        if cache is not None:                                          # KV cache: nối key/value của các bước trước
            if "k" in cache:
                k, v = torch.cat([cache["k"], k], dim=2), torch.cat([cache["v"], v], dim=2)
            cache["k"], cache["v"] = k, v
        T_k = k.shape[2]
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.d_k)
        scores = scores.masked_fill(~self.mask[T_k - T:T_k, :T_k], float("-inf"))
        self.weights = torch.softmax(scores, dim=-1)                   # lưu lại để trực quan hoá
        return self.proj((self.weights @ v).transpose(1, 2).reshape(B, T, D))


class Block(nn.Module):
    def __init__(self, d_model, n_heads, context):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, context)
        self.mlp = nn.Sequential(nn.Linear(d_model, 4 * d_model), nn.GELU(), nn.Linear(4 * d_model, d_model))

    def forward(self, x, cache=None):
        x = x + self.attn(self.ln1(x), cache)
        return x + self.mlp(self.ln2(x))


class GPT(nn.Module):
    def __init__(self, vocab, d_model=128, n_heads=4, n_layers=4, context=64):
        super().__init__()
        self.context = context
        self.tok = nn.Embedding(vocab, d_model)
        self.pos = nn.Embedding(context, d_model)
        self.blocks = nn.ModuleList([Block(d_model, n_heads, context) for _ in range(n_layers)])
        self.ln = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab, bias=False)
        self.head.weight = self.tok.weight                             # weight tying
        self.apply(self._init)

    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, 0.0, 0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, caches=None, start=0):
        T = idx.shape[1]
        x = self.tok(idx) + self.pos(torch.arange(start, start + T, device=idx.device))
        for i, block in enumerate(self.blocks):
            x = block(x, None if caches is None else caches[i])
        return self.head(self.ln(x))

## 2. Dữ liệu và kiểm tra tính nhân quả

Cùng ngữ liệu, cùng cách chia (90% đầu train, 10% cuối validation) và cùng cách lấy batch (64 đoạn × 64 ký tự) như mô hình ngôn ngữ ký tự ở chương 30.

In [ ]:
reviews = pd.read_csv("../../data/sample/reviews_vi.csv")["text"].tolist()
corpus = "\n".join(reviews) + "\n"
chars = sorted(set(corpus))
V = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in corpus])
split = int(0.9 * len(data))
train_data, val_data = data[:split], data[split:]


def get_batch(source, g, batch=64, seq=64):
    starts = torch.randint(0, len(source) - seq - 1, (batch,), generator=g)
    x = torch.stack([source[s:s + seq] for s in starts])
    y = torch.stack([source[s + 1:s + seq + 1] for s in starts])
    return x.to(device), y.to(device)


torch.manual_seed(0)
gpt = GPT(V).to(device)
print(f"{len(corpus):,} ký tự | {V} ký tự khác nhau | GPT: {sum(p.numel() for p in gpt.parameters()):,} tham số")

Kiểm tra trước khi train: đổi các token **từ vị trí 40 trở đi**, logits ở các vị trí **0–39** phải giữ nguyên. Nếu quên mask (hoặc mask sai), mô hình nhìn trộm token cần dự đoán: loss train giảm về gần 0 một cách "kỳ diệu" nhưng sinh văn bản vô nghĩa (bài tập 7).

In [ ]:
def causality_gap(model, pos=40):
    x, _ = get_batch(val_data, torch.Generator().manual_seed(5), batch=2)
    x2 = x.clone()
    x2[:, pos:] = (x2[:, pos:] + 7) % V
    model.eval()
    with torch.no_grad():
        a, b = model(x), model(x2)
    return (a[:, :pos] - b[:, :pos]).abs().max().item(), (a[:, pos:] - b[:, pos:]).abs().max().item()


before, after = causality_gap(gpt)
print(f"thay đổi logits ở vị trí < 40: {before:.1e} | ở vị trí ≥ 40: {after:.2e}")

## 3. Huấn luyện: GPT vs LSTM

Công thức huấn luyện Transformer điển hình: **AdamW**, **warmup** learning rate tuyến tính 100 bước rồi giảm theo **cosine**, gradient clipping. LSTM giữ đúng cấu hình của chương 30 (Adam lr 3e-3). Cả hai: 1 500 bước, đánh giá cross-entropy trên cùng 20 batch validation.

In [ ]:
class CharLSTM(nn.Module):
    def __init__(self, vocab, emb=32, hidden=128):
        super().__init__()
        self.emb = nn.Embedding(vocab, emb)
        self.rnn = nn.LSTM(emb, hidden, batch_first=True)
        self.head = nn.Linear(hidden, vocab)

    def forward(self, x):
        return self.head(self.rnn(self.emb(x))[0])


def evaluate(model, source=None):
    model.eval()
    gv = torch.Generator().manual_seed(123)
    with torch.no_grad():
        return np.mean([nn.functional.cross_entropy(model(x).reshape(-1, V), y.reshape(-1)).item()
                        for x, y in (get_batch(val_data if source is None else source, gv) for _ in range(20))])


def train_lm(model, steps=1500, lr=3e-3, warmup=0, seed=0):
    torch.manual_seed(seed)
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0)
    if warmup:
        schedule = lambda s: min(1.0, (s + 1) / warmup) * 0.5 * (1 + math.cos(math.pi * s / steps))
    else:
        schedule = lambda s: 1.0
    sched = torch.optim.lr_scheduler.LambdaLR(opt, schedule)
    g = torch.Generator().manual_seed(seed)
    t0 = time.perf_counter()
    for _ in range(steps):
        model.train()
        x, y = get_batch(train_data, g)
        opt.zero_grad()
        nn.functional.cross_entropy(model(x).reshape(-1, V), y.reshape(-1)).backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        sched.step()
    return time.perf_counter() - t0


torch.manual_seed(0)
lstm = CharLSTM(V)
t_lstm = train_lm(lstm)
t_gpt = train_lm(gpt, lr=1e-3, warmup=100)
for name, model, t in [("LSTM", lstm, t_lstm), ("GPT", gpt, t_gpt)]:
    val = evaluate(model)
    print(f"{name:4s}: {sum(p.numel() for p in model.parameters()):7,d} tham số | cross-entropy validation = {val:.3f} nats/ký tự "
          f"(perplexity {np.exp(val):.2f}) | {t:.0f}s")
print("kiểm tra nhân quả sau train:", f"{causality_gap(gpt)[0]:.1e}")

Kết quả **gần như bằng nhau**. Không có gì sai: ngữ liệu gồm các review ngắn (trung bình 40 ký tự), sinh từ một số mẫu câu — ngữ cảnh cần thiết ngắn và cả hai mô hình đã chạm gần "đáy" của dữ liệu. Ở quy mô này GPT còn có nhiều tham số hơn ~8 lần và chạy chậm hơn (LSTM dùng kernel cuDNN tối ưu khi có GPU).

Ưu thế của Transformer nằm ở chỗ khác: **song song hoá** toàn bộ chuỗi khi train (không tuần tự theo thời gian), **truy cập trực tiếp** mọi vị trí trong ngữ cảnh, và **mở rộng quy mô** (dữ liệu, tham số, GPU) tốt — thể hiện khi dữ liệu và mô hình lớn. Mục 5 cho thấy một khác biệt **về chất** ngay ở quy mô nhỏ.

## 4. Lấy mẫu: nhiệt độ, top-k, top-p

Từ logits $\mathbf z$ của token tiếp theo:
- **Nhiệt độ** $\tau$: $\text{softmax}(\mathbf z/\tau)$ — $\tau < 1$ sắc hơn (an toàn, lặp), $\tau > 1$ phẳng hơn (đa dạng, nhiều lỗi).
- **Top-$k$**: chỉ giữ $k$ token xác suất cao nhất.
- **Top-$p$** (nucleus, Holtzman et al., 2020): giữ tập **nhỏ nhất** các token có tổng xác suất $\ge p$ — số ứng viên tự co giãn theo độ chắc chắn của mô hình.

In [ ]:
def filter_logits(logits, temperature=1.0, top_k=None, top_p=None):
    logits = logits / temperature
    if top_k is not None:
        kth = torch.topk(logits, top_k, dim=-1).values[..., -1:]
        logits = logits.masked_fill(logits < kth, float("-inf"))
    if top_p is not None:
        sorted_logits, order = torch.sort(logits, descending=True, dim=-1)
        cum = torch.softmax(sorted_logits, dim=-1).cumsum(dim=-1)
        remove = cum - torch.softmax(sorted_logits, dim=-1) >= top_p   # bỏ token khi tổng TRƯỚC nó đã ≥ p
        sorted_logits = sorted_logits.masked_fill(remove, float("-inf"))
        logits = torch.full_like(logits, float("-inf")).scatter(-1, order, sorted_logits)
    return logits


z = torch.tensor([3.0, 2.0, 1.0, 0.0, -1.0])
for kw in [dict(), dict(temperature=0.5), dict(temperature=2.0), dict(top_k=2), dict(top_p=0.9)]:
    print(f"{str(kw):22s} → xác suất {torch.softmax(filter_logits(z, **kw), -1).numpy().round(3)}")

In [ ]:
@torch.no_grad()
def generate(model, prompt="\n", n=150, use_cache=True, seed=0, **sampling):
    model.eval()
    g = torch.Generator(device=device).manual_seed(seed)
    idx = torch.tensor([[stoi[c] for c in prompt]], device=device)
    caches = [dict() for _ in model.blocks] if use_cache else None
    logits = model(idx[:, -model.context:], caches)[:, -1]
    for _ in range(n):
        nxt = torch.multinomial(torch.softmax(filter_logits(logits, **sampling), -1), 1, generator=g)
        idx = torch.cat([idx, nxt], dim=1)
        if use_cache and idx.shape[1] <= model.context:
            logits = model(nxt, caches, start=idx.shape[1] - 1)[:, -1]   # chỉ tính cho token mới
        else:
            logits = model(idx[:, -model.context:])[:, -1]               # không cache, hoặc vượt ngữ cảnh: tính lại cửa sổ cuối
    return "".join(chars[i] for i in idx[0].tolist())


for kw in [dict(temperature=0.5), dict(temperature=1.0), dict(temperature=1.5), dict(temperature=1.5, top_p=0.8)]:
    print(f"── {kw}")
    print(generate(gpt, n=120, **kw).strip())

Nhiệt độ thấp: câu đúng mẫu, lặp lại; nhiệt độ cao: bắt đầu ghép sai từ, sai dấu. Thêm top-$p$ ở cùng nhiệt độ 1.5 cắt bỏ phần đuôi xác suất thấp — nơi sinh ra các lỗi đó. (Cùng `seed` nên khi bộ lọc không làm đổi phân phối ở một bước, ký tự lấy được giống hệt.) Mô hình chỉ học **ngữ liệu nhỏ này** — nó tái tạo các mẫu câu review, không "hiểu" tiếng Việt.

## 5. Transformer mạnh ở đâu: sao chép trong ngữ cảnh

Bài toán: chuỗi 48 token gồm 24 token **ngẫu nhiên** (từ 16 ký hiệu) rồi **lặp lại đúng** 24 token đó. Nửa đầu không thể dự đoán (loss tối thiểu $\ln 16 \approx 2.77$); nửa sau dự đoán **hoàn hảo được** nếu mô hình biết "nhìn lại" và sao chép. Đây là phiên bản đơn giản của khả năng học trong ngữ cảnh (in-context) của mô hình ngôn ngữ — lặp lại tên riêng, biến trong code, mẫu trong prompt.

In [ ]:
VOCAB_COPY, SEG = 16, 24


def copy_batch(n, g):
    a = torch.randint(0, VOCAB_COPY, (n, SEG), generator=g)
    s = torch.cat([a, a], dim=1)
    return s[:, :-1].to(device), s[:, 1:].to(device)


def train_copy(model, steps=3000, seed=0):
    torch.manual_seed(seed)
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    g = torch.Generator().manual_seed(seed)
    xv, yv = copy_batch(1000, torch.Generator().manual_seed(77))
    curve = []
    for step in range(1, steps + 1):
        model.train()
        x, y = copy_batch(64, g)
        opt.zero_grad()
        nn.functional.cross_entropy(model(x).reshape(-1, VOCAB_COPY), y.reshape(-1)).backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        if step % 250 == 0:
            model.eval()
            with torch.no_grad():
                loss = nn.functional.cross_entropy(model(xv).transpose(1, 2), yv, reduction="none")
            curve.append((step, loss[:, :SEG - 1].mean().item(), loss[:, SEG:].mean().item()))
    return curve


copy_curves = {}
for name, make in [("LSTM hidden 128", lambda: CharLSTM(VOCAB_COPY, hidden=128)),
                   ("LSTM hidden 512", lambda: CharLSTM(VOCAB_COPY, hidden=512)),
                   ("GPT 1 khối", lambda: GPT(VOCAB_COPY, d_model=64, n_layers=1, context=2 * SEG))]:
    torch.manual_seed(0)
    model = make()
    t0 = time.perf_counter()
    copy_curves[name] = train_copy(model)
    if name.startswith("GPT"):
        copy_gpt = model
    _, first, second = copy_curves[name][-1]
    print(f"{name:16s}: {sum(p.numel() for p in model.parameters()):9,d} tham số | loss nửa đầu = {first:.3f} | "
          f"loss nửa sau (phần lặp) = {second:.4f} | {time.perf_counter() - t0:.0f}s")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for name, curve in copy_curves.items():
    axes[0].plot([c[0] for c in curve], [c[2] for c in curve], "o-", label=name)
axes[0].axhline(np.log(VOCAB_COPY), color="gray", ls=":", label="ln 16 (đoán ngẫu nhiên)")
axes[0].set_xlabel("bước")
axes[0].set_ylabel("loss trên nửa lặp lại")
axes[0].set_yscale("log")
axes[0].legend()

x, _ = copy_batch(1, torch.Generator().manual_seed(3))
copy_gpt.eval()
with torch.no_grad():
    copy_gpt(x)
A = copy_gpt.blocks[0].attn.weights[0].mean(0).cpu()                    # trung bình các đầu
axes[1].imshow(A, cmap="Blues")
axes[1].set_xlabel("vị trí được nhìn")
axes[1].set_ylabel("vị trí đang dự đoán")
axes[1].set_title("attention của GPT (trung bình 4 đầu)")
look_back = torch.arange(SEG, 2 * SEG - 1) - A[SEG:2 * SEG - 1].argmax(1)
print("phần lặp: vị trí đang dự đoán − vị trí được nhìn nhiều nhất =", look_back.tolist())
plt.tight_layout()
plt.show()

- LSTM (kể cả hidden 512, hơn 1 triệu tham số) **đứng yên ở mức đoán ngẫu nhiên** trên phần lặp trong cả 3 000 bước: muốn sao chép, nó phải **nén** 24 token ngẫu nhiên vào trạng thái ẩn rồi giải nén theo đúng thứ tự — rất khó học bằng gradient.
- GPT **1 khối**, ít tham số hơn LSTM nhỏ, đưa loss phần lặp về gần 0 chỉ sau vài trăm bước. Ma trận attention cho thấy một **đường chéo song song cách 23 vị trí**: khi dự đoán token ở vị trí $t$ trong nửa sau, mô hình nhìn đúng vị trí $t - 23$ — tức token đứng **ngay sau** lần xuất hiện trước của token hiện tại.

Ở đây khoảng cách lặp luôn cố định (24), nên một khối với position embedding là đủ: nhìn theo **vị trí**. Khi khoảng cách thay đổi, mô hình phải nhìn theo **nội dung** ("tìm lần trước token hiện tại xuất hiện, lấy token đứng sau nó") — cơ chế **induction head** cần ít nhất 2 khối (Olsson et al., 2022) — bài tập 8.

## 6. KV cache

Khi sinh token thứ $t$, cách ngây thơ chạy lại mô hình trên **toàn bộ** $t$ token → tổng chi phí sinh $n$ token $\propto n^2$ lượt token qua mạng. Nhưng key/value của các token cũ **không đổi** (causal mask: token cũ không nhìn token mới) → lưu lại (**KV cache**), mỗi bước chỉ tính cho **1 token mới**.

In [ ]:
prompt_text = "\nGiao hàng nhanh"
out_plain = generate(gpt, prompt_text, n=40, use_cache=False, temperature=0.8, seed=1)
out_cache = generate(gpt, prompt_text, n=40, use_cache=True, temperature=0.8, seed=1)
print("cùng kết quả:", out_plain == out_cache, "|", repr(out_cache))

n_new, n_prompt = 40, len(prompt_text)
plain_tokens = sum(n_prompt + i for i in range(n_new))
print(f"số lượt token đi qua mạng khi sinh {n_new} token: không cache = {plain_tokens:,} | có cache = {n_prompt + n_new - 1:,}")

for use_cache in [False, True]:
    t0 = time.perf_counter()
    for s in range(5):
        generate(gpt, prompt_text, n=40, use_cache=use_cache, seed=s)
    print(f"{'có' if use_cache else 'không':5s} cache: {(time.perf_counter() - t0) / 5 * 1000:.0f} ms / lần sinh 40 token")

Lượng tính toán giảm hơn 20 lần, nhưng thời gian ở đây gần như không đổi: mô hình nhỏ và ngữ cảnh ngắn nên mỗi lần gọi bị chi phối bởi chi phí cố định (Python, khởi chạy kernel GPU) chứ không phải phép nhân ma trận; với LLM thật — ngữ cảnh hàng nghìn token, hàng chục lớp — KV cache là bắt buộc, và **bộ nhớ** của cache (tỉ lệ với số lớp × độ dài × $d_{model}$) trở thành nút thắt khi phục vụ mô hình (các kỹ thuật như multi-query/grouped-query attention giảm kích thước cache).

## 7. Vài con số về quy mô

- **Tham số** của GPT với $L$ khối, $d_{model} = d$: khoảng $12Ld^2$ (attention $4d^2$ + FFN $8d^2$ mỗi khối) cộng embedding.
- **FLOPs huấn luyện** $\approx 6ND$ ($N$ tham số, $D$ token) — 2 cho forward, 4 cho backward.
- **Chinchilla** (Hoffmann et al., 2022): với ngân sách tính toán cố định, tối ưu khi số token ≈ 20 × số tham số.
- **Tokenizer**: GPT thật dùng BPE/subword (chương 32) thay vì ký tự — chuỗi ngắn hơn nhiều lần, ngữ cảnh "chứa" được nhiều nội dung hơn.

In [ ]:
for name, L, d in [("mini GPT bài này", 4, 128), ("GPT-2 small", 12, 768), ("GPT-2 XL", 48, 1600)]:
    print(f"{name:18s}: 12·L·d² ≈ {12 * L * d * d / 1e6:8.1f} triệu tham số (chưa tính embedding)")
N = sum(p.numel() for p in gpt.parameters())
D = 1500 * 64 * 64
print(f"mini GPT: N = {N:,}, D = {D:,} token đã thấy → ≈ {6 * N * D:.1e} FLOPs; Chinchilla gợi ý ≈ {20 * N:,} token cho N này")

## 🧠 Kiểm tra nhanh

1. GPT khác encoder Transformer ở bài 02 ở điểm nào?
2. Làm sao kiểm tra một mô hình ngôn ngữ thực sự nhân quả? Nếu quên causal mask, triệu chứng là gì?
3. Vì sao trên ngữ liệu review, GPT không tốt hơn LSTM? Ưu thế thật của Transformer là gì?
4. Top-$k$ và top-$p$ khác nhau thế nào?
5. KV cache lưu gì? Vì sao kết quả không đổi?
6. Vì sao LSTM không học được bài sao chép còn GPT 1 khối thì được?

## 📌 Tóm tắt

- GPT = embedding token + vị trí → các khối (causal MHA + FFN, pre-LN, residual) → LN → đầu ra tied; train dự đoán token tiếp theo song song mọi vị trí.
- Công thức train: AdamW, warmup + cosine, clipping, khởi tạo nhỏ; luôn kiểm tra tính nhân quả.
- Trên dữ liệu ngắn, đơn giản: GPT ≈ LSTM. Trên bài cần truy cập lại ngữ cảnh xa (sao chép), Transformer vượt hẳn — nền tảng của học trong ngữ cảnh.
- Lấy mẫu: nhiệt độ, top-$k$, top-$p$; KV cache để sinh nhanh; FLOPs ≈ $6ND$.

## 📚 Tài liệu tham khảo

- Radford et al. (2018, 2019) — *Improving Language Understanding by Generative Pre-Training*; *Language Models are Unsupervised Multitask Learners* (GPT, GPT-2)
- Holtzman et al. (2020) — *The Curious Case of Neural Text Degeneration* (top-p)
- Olsson et al. (2022) — *In-context Learning and Induction Heads* (Anthropic)
- Kaplan et al. (2020) — *Scaling Laws for Neural Language Models*; Hoffmann et al. (2022) — *Training Compute-Optimal Large Language Models*
- Karpathy — [nanoGPT](https://github.com/karpathy/nanoGPT), video "Let's build GPT: from scratch, in code, spelled out"